# KeyGuard AI — LSTM Training

Trains the keystroke-dynamics anomaly-detection LSTM.
Produces a checkpoint file (`model.pt`) with the exact input contract `model.extract_features` produces at inference time:

`(SEQUENCE_LENGTH=50, FEATURE_DIM=4)` per sample — `[dwell_time, flight_time, rolling_speed, key_norm]`.

Output: a single-value sigmoid authenticity score in `[0, 1]` per sequence.

In [ ]:
!pip install -q torch numpy

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEQUENCE_LENGTH = 50
FEATURE_DIM = 4
ROLLING_WINDOW_SECONDS = 2.0

class KeystrokeLSTM(nn.Module):
    def __init__(self, sequence_length=50, feature_dim=4):
        super().__init__()
        self.lstm1 = nn.LSTM(input_size=feature_dim, hidden_size=64, batch_first=True)
        self.lstm2 = nn.LSTM(input_size=64, hidden_size=32, batch_first=True)
        self.fc1 = nn.Linear(32, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.lstm1(x)
        out, _ = self.lstm2(out)
        out = out[:, -1, :]
        out = self.relu(self.fc1(out))
        out = self.sigmoid(self.fc2(out))
        return out

In [ ]:
def generate_session(profile: str, seq_len: int = 50, window: float = 2.0) -> np.ndarray:
    pad_len = np.random.randint(5, 35) if np.random.rand() < 0.65 else 0
    real_len = seq_len - pad_len

    if profile == 'genuine':
        dwells = np.clip(np.random.normal(0.10, 0.025, real_len), 0.04, 0.22)
        flights = np.clip(np.random.normal(0.14, 0.04, real_len), 0.05, 0.35)
    elif profile == 'bot':
        dwells = np.clip(np.random.exponential(0.002, real_len), 0.0001, 0.01)
        flights = np.clip(np.random.exponential(0.002, real_len), 0.0001, 0.01)
    elif profile == 'erratic':
        dwells = np.clip(np.random.normal(0.40, 0.15, real_len), 0.15, 0.90)
        flights = np.clip(np.random.exponential(0.7, real_len) + 0.3, 0.2, 2.5)
    else:
        dwells = np.random.uniform(0.001, 0.8, real_len)
        flights = np.random.uniform(0.001, 1.5, real_len)

    intervals = dwells + flights
    timestamps = np.cumsum(intervals)
    speeds = np.zeros(real_len, dtype=np.float32)
    for i, t in enumerate(timestamps):
        speeds[i] = np.sum(timestamps[:i + 1] >= (t - window)) / window

    keys = np.random.uniform(0.1, 0.9, real_len)
    real_features = np.column_stack([dwells, flights, speeds, keys]).astype(np.float32)

    if pad_len > 0:
        pad = np.zeros((pad_len, FEATURE_DIM), dtype=np.float32)
        return np.vstack([pad, real_features])
    return real_features

def generate_synthetic_data(num_samples=2000):
    half = num_samples // 2
    X_genuine = [generate_session('genuine') for _ in range(half)]
    y_genuine = np.ones((half, 1), dtype=np.float32)
    anom_profiles = ['bot', 'erratic', 'noise']
    X_anomaly = [generate_session(anom_profiles[i % 3]) for i in range(half)]
    y_anomaly = np.zeros((half, 1), dtype=np.float32)
    X = np.array(X_genuine + X_anomaly, dtype=np.float32)
    y = np.vstack([y_genuine, y_anomaly]).astype(np.float32)
    indices = np.random.permutation(num_samples)
    return X[indices], y[indices]

X, y = generate_synthetic_data(2000)
print('Generated dataset shape:', X.shape, y.shape)

In [ ]:
val_split = int(0.8 * len(X))
train_dataset = TensorDataset(torch.from_numpy(X[:val_split]), torch.from_numpy(y[:val_split]))
val_dataset = TensorDataset(torch.from_numpy(X[val_split:]), torch.from_numpy(y[val_split:]))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

model = KeystrokeLSTM(SEQUENCE_LENGTH, FEATURE_DIM)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

for epoch in range(1, 21):
    model.train()
    for bx, by in train_loader:
        optimizer.zero_grad()
        pred = model(bx)
        loss = criterion(pred, by)
        loss.backward()
        optimizer.step()
    if epoch % 5 == 0:
        print(f'Epoch {epoch}/20 completed.')

In [ ]:
model.eval()
dummy_input = torch.zeros((1, SEQUENCE_LENGTH, FEATURE_DIM), dtype=torch.float32)
traced_model = torch.jit.trace(model, dummy_input)
torch.jit.save(traced_model, 'model.pt')
print('Saved model.pt successfully!')